# Ollama — LLM local con Python

Ollama permite ejecutar modelos LLM open source directamente en local,
sin API key y sin mandar datos a servidores externos.
Todo ocurre en tu máquina.

En este notebook conectamos Python con Ollama via HTTP,
exactamente igual que con la API de Anthropic / OpenAI —
la diferencia es que el endpoint es `localhost`.

## Qué veremos
- Verificar que Ollama está corriendo y qué modelos tenemos disponibles
- Hacer nuestra primera llamada al modelo desde Python
- Comparar la interfaz con el SDK de Anthropic que ya conocemos


In [1]:
import httpx

# Verificamos que Ollama está corriendo
response = httpx.get("http://localhost:11434/api/tags")
print(response.json())

{'models': [{'name': 'qwen2.5:7b', 'model': 'qwen2.5:7b', 'modified_at': '2026-05-03T13:57:09.685685369+02:00', 'size': 4683087332, 'digest': '845dbda0ea48ed749caafd9e6037047aa19acfcfd82e704d7ca97d631a0b697e', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen2', 'families': ['qwen2'], 'parameter_size': '7.6B', 'quantization_level': 'Q4_K_M'}}]}


## Primera llamada al modelo

Hacemos un POST a la API de Ollama con el mensaje del usuario.
La estructura es idéntica a la API de Anthropic: `model`, `messages` con `role` y `content`.
`stream: False` indica que queremos la respuesta completa de golpe, no token a token.
El resultado llega en `data["message"]["content"]`.

In [8]:
response = httpx.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen2.5:7b",
        "messages": [{"role": "user", "content": "Explica qué es un LLM en 2 frases"}],
        "stream": False
    },
    timeout=60
)

data = response.json()
print(f"RAW: {data}")
print("\n")
print(data["message"]["content"])

RAW: {'model': 'qwen2.5:7b', 'created_at': '2026-05-03T12:37:34.544625Z', 'message': {'role': 'assistant', 'content': 'Un LLM es una Arquitectura de Modelo de Lenguaje Gigante (en inglés, Large Language Model), que se refiere a modelos de inteligencia artificial capaces de generar texto humanoide y entender una amplia gama de temas. Estos modelos están entrenados en grandes conjuntos de datos para realizar diversas tareas relacionadas con el lenguaje natural.'}, 'done': True, 'done_reason': 'stop', 'total_duration': 3138486333, 'load_duration': 58666708, 'prompt_eval_count': 41, 'prompt_eval_duration': 52483708, 'eval_count': 79, 'eval_duration': 2998402788}


Un LLM es una Arquitectura de Modelo de Lenguaje Gigante (en inglés, Large Language Model), que se refiere a modelos de inteligencia artificial capaces de generar texto humanoide y entender una amplia gama de temas. Estos modelos están entrenados en grandes conjuntos de datos para realizar diversas tareas relacionadas con el leng

## Analizando la respuesta

La API devuelve metadatos además del texto: duración total, tokens procesados,
tokens generados. Esto es útil para medir rendimiento y coste computacional.
Calculamos los tokens por segundo para benchmarking.

In [9]:
tokens = data["eval_count"]
duration_s = data["eval_duration"] / 1e9  # nanosegundos a segundos

print(f"Respuesta: {data['message']['content']}")
print(f"\nTokens generados: {tokens}")
print(f"Duración: {duration_s:.2f}s")
print(f"Velocidad: {tokens / duration_s:.1f} tokens/segundo")

Respuesta: Un LLM es una Arquitectura de Modelo de Lenguaje Gigante (en inglés, Large Language Model), que se refiere a modelos de inteligencia artificial capaces de generar texto humanoide y entender una amplia gama de temas. Estos modelos están entrenados en grandes conjuntos de datos para realizar diversas tareas relacionadas con el lenguaje natural.

Tokens generados: 79
Duración: 3.00s
Velocidad: 26.3 tokens/segundo
